# Project: SVM with Pegasos 🛡️
Train a linear **support vector machine** with **Pegasos**, a stochastic sub-gradient method on the hinge loss (Shalev-Shwartz et al. 2007), which is what makes SVMs scale to millions of examples. Compare it with scikit-learn's `LinearSVC` on breast-cancer data and find the support vectors.

Topic: [[Support Vector Machines]], [[Kernel Methods]] · guide note: [[Project - SVM with Pegasos]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
data = load_breast_cancer(); X = (data.data - data.data.mean(0)) / data.data.std(0); y = np.where(data.target == 1, 1, -1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

## 1. The SVM objective\n$\frac{\lambda}{2}\lVert w\rVert^2 + \frac1n\sum_i \max(0, 1 - y_i(w^\top x_i + b))$: a margin term plus the hinge loss, which is zero for points beyond the margin.

In [ ]:
def svm_objective(w, b, X, y, lam):
    # TODO 1: regulariser + mean hinge loss
    raise NotImplementedError  # TODO 1

## 2. Pegasos\nFor $t = 1 \dots T$: pick a random example $i$, step size $\eta_t = 1/(\lambda t)$. If $y_i(w^\top x_i+b) < 1$ (inside the margin): $w \leftarrow (1-\eta_t\lambda)w + \eta_t y_ix_i$, $b \leftarrow b + \eta_t y_i$; else just shrink $w \leftarrow (1-\eta_t\lambda)w$. Run `epochs × n` steps; return `(w, b)`.

In [ ]:
def pegasos(X, y, lam=0.01, epochs=20, seed=0):
    # TODO 2: stochastic sub-gradient descent on the SVM objective
    raise NotImplementedError  # TODO 2

In [ ]:
check("objective", lambda: abs(svm_objective(np.array([1.0, 0]), 0.0, np.array([[2.0, 0], [0.5, 0], [-1, 0]]), np.array([1, 1, 1]), 0.1) - (0.05 + (0 + 0.5 + 2) / 3)) < 1e-12,
      "Margins 2, 0.5, −1 → hinge 0, 0.5, 2; plus λ/2·‖w‖² = 0.05.")
if ready("objective"):
    try:
        LAM = 0.1
        w, b_ = pegasos(X_tr, y_tr, lam=LAM); acc = np.mean(np.sign(X_te @ w + b_) == y_te)
        sk = LinearSVC(C=1 / (LAM * len(X_tr)), loss="hinge", max_iter=50000).fit(X_tr, y_tr)
        print(f"Pegasos test accuracy {acc:.3f} | LinearSVC {sk.score(X_te, y_te):.3f}")
        print(f"objective: Pegasos {svm_objective(w, b_, X_tr, y_tr, LAM):.4f}  LinearSVC {svm_objective(sk.coef_[0], sk.intercept_[0], X_tr, y_tr, LAM):.4f}")
        m = y_tr * (X_tr @ w + b_); print(f"support vectors (margin ≤ 1): {np.sum(m <= 1)} of {len(X_tr)} training points")
        plt.hist(m, 50); plt.axvline(1, c="r"); plt.xlabel("y·f(x)"); plt.title("functional margins: only points left of the red line shape w"); plt.show()
        check("pegasos works", lambda: acc > 0.94 and svm_objective(w, b_, X_tr, y_tr, LAM) < 1.1 * svm_objective(sk.coef_[0], sk.intercept_[0], X_tr, y_tr, LAM),
              "Accuracy > 94 % and an objective within 10 % of LinearSVC's.")
        print("→ Try LAM = 0.01: the step size 1/(λt) is then 10× larger at the start and Pegasos needs ~5× more epochs to reach the optimum.")
    except NotImplementedError:
        print("❌ pegasos works: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def svm_objective(w, b, X, y, lam):
    margins = y * (X @ w + b)
    return float(lam / 2 * w @ w + np.mean(np.maximum(0, 1 - margins)))
```

```python
def pegasos(X, y, lam=0.01, epochs=20, seed=0):
    rng = np.random.default_rng(seed); n, d = X.shape
    w, b = np.zeros(d), 0.0
    for t in range(1, epochs * n + 1):
        i = rng.integers(n); eta = 1 / (lam * t)
        if y[i] * (X[i] @ w + b) < 1:
            w = (1 - eta * lam) * w + eta * y[i] * X[i]; b += eta * y[i]
        else:
            w = (1 - eta * lam) * w
    return w, b
```
</details>

## Stretch goals
- Kernelise Pegasos with an RBF kernel (keep a count per training point instead of `w`).
- Run λ = 0.01 for 20 vs. 100 epochs and compare the objective with LinearSVC's (convergence speed depends on λ).
- Plot accuracy vs. λ: small λ = narrow margin (overfit), large λ = wide margin (underfit).
- Compare with logistic regression: same linear model, different loss ([[Logistic Regression]]).